# Phase 3 — Calibrate & validate (CPU, no GPU/model)

Runs on the graded cache from Phase 2 (`claims_phase2.jsonl`). Computes the Gate 2 AUROC, then the two conformal results:
- **Global CRC** — one threshold, controls the *marginal* retained-hallucination risk.
- **Severity-aware CRC** — a stricter risk budget on `dangerous` claims than on `benign` ones (the paper's contribution).

Everything here is seconds of CPU. Gate 1 (labeler-vs-physician agreement) is an optional last cell that needs OpenAI.

## 1. Config + load graded claims

In [ ]:
# ---------- Config ----------
SAC_PATH        = "/kaggle/input/severity-aware-conformal"   # folder with the sac/ package
CACHE           = "/kaggle/working/claims_phase2.jsonl"      # graded cache from Phase 2
ALPHA_MARGINAL  = 0.10    # global risk budget
ALPHA_DANGEROUS = 0.05    # stricter budget for dangerous claims
ALPHA_BENIGN    = 0.15    # looser budget for benign claims
N_SPLITS        = 300     # random cal/test splits to average over
# ----------------------------

import os, sys
# auto-fallback to local paths when run from notebooks/ outside Kaggle
if not os.path.isdir(os.path.join(SAC_PATH, "sac")): SAC_PATH = ".."
if not os.path.exists(CACHE):                         CACHE = "../claims_phase2.jsonl"
sys.path.insert(0, SAC_PATH)

import numpy as np
from sac.cache import load_claims
from sac.validate import score_auroc, gate2_pass, unverifiable_count
from sac.crc import (crc_calibrate, crc_calibrate_stratified, apply_global, apply_stratified,
                     retention, realized_risk_marginal, tier_risk_marginal)

claims = load_claims(CACHE)
verifiable = [c for c in claims if c.label in (0, 1)]   # CRC ignores unverifiable (-1)
print(f"{len(claims)} claims | {len(verifiable)} verifiable | "
      f"{unverifiable_count(claims)} unverifiable (excluded)")

## 2. Gate 2 — does the score separate truth from hallucination? (AUROC ≥ 0.70)

In [ ]:
au = score_auroc(claims)
print(f"GATE 2  P(true) AUROC = {au:.3f}  ->  {'PASS' if gate2_pass(au) else 'FAIL (<0.70 — fix score)'}")

## 3. One split — global vs severity-aware thresholds

Illustrative: split verifiable claims 50/50, calibrate on one half, measure on the other. Note the severity-aware run picks a **higher** threshold for dangerous claims and a lower one for benign.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(len(verifiable)); k = len(verifiable) // 2
calib = [verifiable[i] for i in idx[:k]]
test  = [verifiable[i] for i in idx[k:]]

# Global: one threshold for the marginal risk budget
lam = crc_calibrate([c.confidence for c in calib], [c.label for c in calib], ALPHA_MARGINAL)
kept_g = apply_global(test, lam)
print(f"GLOBAL      lambda={lam:.3f}")
print(f"  marginal risk ={realized_risk_marginal(test, kept_g):.3f} (target {ALPHA_MARGINAL})"
      f"  dangerous risk={tier_risk_marginal(test, kept_g, 'dangerous'):.3f}"
      f"  retention={retention(test, kept_g):.3f}")

# Severity-aware: per-tier thresholds
th = crc_calibrate_stratified(calib, {"dangerous": ALPHA_DANGEROUS, "benign": ALPHA_BENIGN})
kept_s = apply_stratified(test, th)
print(f"\nSEVERITY-AWARE  thresholds={{'dangerous': {th['dangerous']:.3f}, 'benign': {th['benign']:.3f}}}")
print(f"  dangerous risk={tier_risk_marginal(test, kept_s, 'dangerous'):.3f} (target {ALPHA_DANGEROUS})"
      f"  benign risk={tier_risk_marginal(test, kept_s, 'benign'):.3f} (target {ALPHA_BENIGN})"
      f"  retention={retention(test, kept_s):.3f}")

## 4. Headline — risk & retention averaged over many splits

One split is noisy (only ~80 hallucinations). Averaging over `N_SPLITS` random calibration/test splits gives the paper's main result: severity-aware CRC drives down **dangerous-claim** risk where global CRC does not, at minimal retention cost.

In [ ]:
g = {"dang": [], "ben": [], "ret": [], "marg": []}
s = {"dang": [], "ben": [], "ret": []}
for seed in range(N_SPLITS):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(verifiable)); k = len(verifiable) // 2
    cal = [verifiable[i] for i in idx[:k]]
    te  = [verifiable[i] for i in idx[k:]]

    lam = crc_calibrate([c.confidence for c in cal], [c.label for c in cal], ALPHA_MARGINAL)
    kg = apply_global(te, lam)
    g["marg"].append(realized_risk_marginal(te, kg)); g["dang"].append(tier_risk_marginal(te, kg, "dangerous"))
    g["ben"].append(tier_risk_marginal(te, kg, "benign")); g["ret"].append(retention(te, kg))

    th = crc_calibrate_stratified(cal, {"dangerous": ALPHA_DANGEROUS, "benign": ALPHA_BENIGN})
    ks = apply_stratified(te, th)
    s["dang"].append(tier_risk_marginal(te, ks, "dangerous")); s["ben"].append(tier_risk_marginal(te, ks, "benign"))
    s["ret"].append(retention(te, ks))

m = np.mean
print(f"Averaged over {N_SPLITS} splits  "
      f"(alpha: marginal={ALPHA_MARGINAL}, dangerous={ALPHA_DANGEROUS}, benign={ALPHA_BENIGN})\n")
print(f"                    GLOBAL    SEVERITY-AWARE")
print(f"  dangerous risk     {m(g['dang']):.3f}      {m(s['dang']):.3f}     (target {ALPHA_DANGEROUS})")
print(f"  benign risk        {m(g['ben']):.3f}      {m(s['ben']):.3f}     (target {ALPHA_BENIGN})")
print(f"  retention (true)   {m(g['ret']):.3f}      {m(s['ret']):.3f}")
print(f"  marginal risk      {m(g['marg']):.3f}      --        (target {ALPHA_MARGINAL})")
print(f"\n  P(dangerous risk > {ALPHA_DANGEROUS}):  global={m(np.array(g['dang'])>ALPHA_DANGEROUS):.2f}   "
      f"severity-aware={m(np.array(s['dang'])>ALPHA_DANGEROUS):.2f}")
print("  ^ how often each method violates the danger budget — the case for severity-awareness.")

## 5. (Optional) Gate 1 — does our judge agree with physicians?

Validates the OpenAI grader against K-QA's physician NLI annotations. **Needs internet + an OpenAI key**, and makes one API call per pair (subsampled below to keep it cheap). Skip unless you want the agreement number for the paper.

In [ ]:
N_NLI = 200   # subsample of physician-annotated pairs to grade (set None for all ~1k)

!wget -q -O /kaggle/working/nli.csv https://raw.githubusercontent.com/Itaymanes/K-QA/main/dataset/NLI_medical_annotator.csv
NLI_PATH = "/kaggle/working/nli.csv" if os.path.exists("/kaggle/working/nli.csv") else "nli.csv"

from kaggle_secrets import UserSecretsClient
os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")

from sac.kqa_loader import load_physician_nli
from sac.validate import labeler_agreement
from sac.openai_judge import OpenAIJudge

pairs = load_physician_nli(NLI_PATH)
if N_NLI:
    pairs = pairs[:N_NLI]
g1 = labeler_agreement(pairs, OpenAIJudge(model="gpt-4o"))
print(f"GATE 1  labeler accuracy={g1['accuracy']:.3f}  kappa={g1['kappa']:.3f}  n={g1['n']}")